In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

print(dataset_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1


In [3]:
drivers = sorted(

    d

    for d in os.listdir(dataset_path)

    if os.path.isdir(os.path.join(dataset_path, d))

    and d.startswith("D")

)

print(drivers)

['D1', 'D2', 'D3', 'D4', 'D5', 'D6']


In [4]:
trip_list = []

for driver in drivers:

    driver_path = os.path.join(
        dataset_path,
        driver
    )

    trips = sorted(

        trip

        for trip in os.listdir(driver_path)

        if os.path.isdir(
            os.path.join(driver_path, trip)
        )

    )

    for trip in trips:

        trip_list.append({

            "driver": driver,

            "trip": trip

        })

print(len(trip_list))

trip_list[:5]

40


[{'driver': 'D1', 'trip': '20151110175712-16km-D1-NORMAL1-SECONDARY'},
 {'driver': 'D1', 'trip': '20151110180824-16km-D1-NORMAL2-SECONDARY'},
 {'driver': 'D1', 'trip': '20151111123124-25km-D1-NORMAL-MOTORWAY'},
 {'driver': 'D1', 'trip': '20151111125233-24km-D1-AGGRESSIVE-MOTORWAY'},
 {'driver': 'D1', 'trip': '20151111132348-25km-D1-DROWSY-MOTORWAY'}]

In [5]:
from sklearn.model_selection import train_test_split

train_trips, test_trips = train_test_split(

    trip_list,

    test_size=0.20,

    random_state=42

)

print(len(train_trips))
print(len(test_trips))

32
8


In [6]:
def load_gps(gps_path):

    gps_columns = [

        "timestamp",
        "speed",
        "latitude",
        "longitude",
        "altitude",

        "gps_quality",
        "satellites",

        "heading",

        "extra_1",
        "extra_2",
        "extra_3",
        "extra_4"

    ]

    gps_df = pd.read_csv(
        gps_path,
        sep=r"\s+",
        header=None,
        names=gps_columns
    )

    return gps_df

def synchronize_sensors(acc_df, gps_df):

    master_df = pd.merge_asof(

        acc_df.sort_values("timestamp"),

        gps_df.sort_values("timestamp"),

        on="timestamp",

        direction="nearest"

    )

    return master_df

def engineer_features(master_df):

    master_df = master_df.copy()

    # -------------------------------------------------
    # Acceleration Features
    # -------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # -------------------------------------------------
    # Delta Features
    # -------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    return master_df

def extract_statistics(signal):

    features = {}

    features["mean"] = signal.mean()

    features["std"] = signal.std()

    features["min"] = signal.min()

    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal**2)
    )

    return features

def extract_window_features(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics(window[feature])

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

def create_sliding_windows(
        feature_df,
        feature_list,
        window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features(
            window,
            feature_list
        )

        all_window_features.append(window_stats)

    return pd.DataFrame(all_window_features)

def load_accelerometer(acc_path):

    acc_columns = [

        "timestamp",
        "active",

        "acc_x",
        "acc_y",
        "acc_z",

        "acc_x_kf",
        "acc_y_kf",
        "acc_z_kf",

        "roll",
        "pitch",
        "yaw"

    ]

    acc_df = pd.read_csv(
        acc_path,
        sep=r"\s+",
        header=None,
        names=acc_columns
    )

    return acc_df
def extract_statistics_v2(signal):

    features = {}

    # ------------------------------
    # Basic Statistics
    # ------------------------------

    features["mean"] = signal.mean()
    features["std"] = signal.std()
    features["variance"] = signal.var()

    features["min"] = signal.min()
    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal ** 2)
    )

    # ------------------------------
    # Distribution Shape
    # ------------------------------

    features["skewness"] = signal.skew()

    features["kurtosis"] = signal.kurt()

    # ------------------------------
    # Percentiles
    # ------------------------------

    features["q25"] = signal.quantile(0.25)

    features["q75"] = signal.quantile(0.75)

    features["iqr"] = (
        features["q75"] -
        features["q25"]
    )

    return features

def extract_window_features_v2(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics_v2(window[feature])

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats
def create_sliding_windows_v2(
    feature_df,
    feature_list,
    window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features_v2(
            window,
            feature_list
        )

        all_window_features.append(
            window_stats
        )

    return pd.DataFrame(
        all_window_features
    )



In [17]:
def parse_trip_info(trip_name):

    parts = trip_name.split("-")

    return {
        "date": parts[0],
        "distance": parts[1],
        "driver": parts[2],
        "behavior": parts[3],
        "road_type": parts[4]
    }
    def simplify_behavior(label):

     if "NORMAL" in label:
        return "NORMAL"

    if "AGGRESSIVE" in label:
        return "AGGRESSIVE"

    if "DROWSY" in label:
        return "DROWSY"

    return label

def process_trip_v2(
    dataset_path,
    driver,
    trip,
    window_size
):

    trip_path = os.path.join(
        dataset_path,
        driver,
        trip
    )

    acc_path = os.path.join(
        trip_path,
        "RAW_ACCELEROMETERS.txt"
    )

    gps_path = os.path.join(
        trip_path,
        "RAW_GPS.txt"
    )

    # Load Sensors
    acc_df = load_accelerometer(acc_path)
    gps_df = load_gps(gps_path)

    # Synchronize
    master_df = synchronize_sensors(
        acc_df,
        gps_df
    )

    # Feature Engineering
    feature_df = engineer_features(master_df)

    # Advanced Sliding Window
    window_dataset = create_sliding_windows_v2(
        feature_df,
        window_features,
        window_size
    )

    # Labels
    info = parse_trip_info(trip)

    window_dataset["driver"] = info["driver"]
    window_dataset["road_type"] = info["road_type"]
    window_dataset["behavior"] = simplify_behavior(
        info["behavior"]
    )

    return window_dataset
def simplify_behavior(label):

    if "NORMAL" in label:
        return "NORMAL"

    if "AGGRESSIVE" in label:
        return "AGGRESSIVE"

    if "DROWSY" in label:
        return "DROWSY"

    return label

In [12]:
import inspect

print(inspect.signature(process_trip_v2))

(dataset_path, driver, trip, window_size)


In [15]:
window_features = [
    "acc_resultant",
    "acc_horizontal",
    "speed",
    "speed_delta",
    "roll",
    "pitch",
    "yaw"
]

In [18]:
sample_dataset = process_trip_v2(
    dataset_path,
    sample_driver,
    sample_trip,
    30
)

print(sample_dataset.shape)

(6141, 87)


In [19]:
WINDOW_SIZE = 60

In [21]:
train_datasets = []

for trip in train_trips:

    print(
        f"Processing Train: "
        f"{trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v2(
        dataset_path,
        trip["driver"],
        trip["trip"],
        WINDOW_SIZE
    )

    train_datasets.append(trip_dataset)

Processing Train: D6 - 20151221120051-26km-D6-AGGRESSIVE-MOTORWAY
Processing Train: D1 - 20151111135612-13km-D1-DROWSY-SECONDARY
Processing Train: D4 - 20151204152848-25km-D4-NORMAL-MOTORWAY
Processing Train: D2 - 20151120135152-25km-D2-DROWSY-MOTORWAY
Processing Train: D2 - 20151120164606-16km-D2-DROWSY-SECONDARY
Processing Train: D5 - 20151211162829-16km-D5-NORMAL1-SECONDARY
Processing Train: D5 - 20151211170502-16km-D5-DROWSY-SECONDARY
Processing Train: D2 - 20151120133502-26km-D2-AGGRESSIVE-MOTORWAY
Processing Train: D3 - 20151126125458-16km-D3-NORMAL2-SECONDARY
Processing Train: D4 - 20151203175637-17km-D4-DROWSY-SECONDARY
Processing Train: D1 - 20151110175712-16km-D1-NORMAL1-SECONDARY
Processing Train: D5 - 20151211165606-12km-D5-AGGRESSIVE-SECONDARY
Processing Train: D1 - 20151111134545-16km-D1-AGGRESSIVE-SECONDARY
Processing Train: D2 - 20151120162105-17km-D2-NORMAL2-SECONDARY
Processing Train: D1 - 20151110180824-16km-D1-NORMAL2-SECONDARY
Processing Train: D5 - 20151209153137-

In [22]:
train_dataset = pd.concat(
    train_datasets,
    ignore_index=True
)

print(train_dataset.shape)

train_dataset.head()

(242965, 87)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,yaw_median,yaw_rms,yaw_skewness,yaw_kurtosis,yaw_q25,yaw_q75,yaw_iqr,driver,road_type,behavior
0,0.062950,0.037494,0.001406,0.012961,0.178804,0.058821,0.073110,0.994585,0.711390,0.033116,...,-0.0285,0.026448,0.142478,-1.728146,-0.03325,-0.01400,0.01925,D6,MOTORWAY,AGGRESSIVE
1,0.060696,0.034344,0.001179,0.012961,0.144686,0.056502,0.069598,0.821860,0.113281,0.033116,...,-0.0270,0.026217,0.081806,-1.758558,-0.03325,-0.01400,0.01925,D6,MOTORWAY,AGGRESSIVE
2,0.060563,0.034495,0.001190,0.012961,0.144686,0.056502,0.069556,0.807773,0.093973,0.033116,...,-0.0250,0.025936,0.018867,-1.769954,-0.03325,-0.01375,0.01950,D6,MOTORWAY,AGGRESSIVE
3,0.059966,0.034357,0.001180,0.012961,0.144686,0.053694,0.068969,0.863860,0.203253,0.033116,...,-0.0230,0.025623,-0.041783,-1.764851,-0.03325,-0.01300,0.02025,D6,MOTORWAY,AGGRESSIVE
4,0.059171,0.033793,0.001142,0.012961,0.144686,0.053694,0.068001,0.933059,0.452223,0.033116,...,-0.0215,0.025261,-0.102163,-1.749063,-0.03225,-0.01300,0.01925,D6,MOTORWAY,AGGRESSIVE


In [23]:
test_datasets = []

for trip in test_trips:

    print(
        f"Processing Test: "
        f"{trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v2(
        dataset_path,
        trip["driver"],
        trip["trip"],
        WINDOW_SIZE
    )

    test_datasets.append(trip_dataset)

Processing Test: D3 - 20151126132013-17km-D3-DROWSY-SECONDARY
Processing Test: D3 - 20151126124208-16km-D3-NORMAL1-SECONDARY
Processing Test: D3 - 20151126113754-26km-D3-DROWSY-MOTORWAY
Processing Test: D4 - 20151204154908-25km-D4-AGGRESSIVE-MOTORWAY
Processing Test: D1 - 20151111132348-25km-D1-DROWSY-MOTORWAY
Processing Test: D2 - 20151120163350-16km-D2-AGGRESSIVE-SECONDARY
Processing Test: D6 - 20151221112434-17km-D6-NORMAL-SECONDARY
Processing Test: D4 - 20151204160823-25km-D4-DROWSY-MOTORWAY


In [24]:
test_dataset = pd.concat(
    test_datasets,
    ignore_index=True
)

print(test_dataset.shape)

test_dataset.head()

(66060, 87)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,yaw_median,yaw_rms,yaw_skewness,yaw_kurtosis,yaw_q25,yaw_q75,yaw_iqr,driver,road_type,behavior
0,0.069472,0.041229,0.001700,0.009487,0.146826,0.056582,0.080609,0.716307,-0.647953,0.040467,...,0.0260,0.028509,0.541718,-0.239422,0.01975,0.03125,0.01150,D3,SECONDARY,DROWSY
1,0.067631,0.040168,0.001613,0.009487,0.146826,0.055759,0.078489,0.781847,-0.482125,0.039345,...,0.0260,0.029255,0.624937,-0.065010,0.02075,0.03200,0.01125,D3,SECONDARY,DROWSY
2,0.066677,0.040049,0.001604,0.009487,0.146826,0.053646,0.077608,0.849096,-0.373484,0.039345,...,0.0265,0.030088,0.740141,0.150049,0.02250,0.03225,0.00975,D3,SECONDARY,DROWSY
3,0.064657,0.038979,0.001519,0.009487,0.146826,0.051527,0.075330,0.909001,-0.183656,0.037702,...,0.0275,0.030945,0.819152,0.281245,0.02375,0.03350,0.00975,D3,SECONDARY,DROWSY
4,0.062385,0.038071,0.001449,0.009487,0.146826,0.050660,0.072919,0.934266,-0.004782,0.035844,...,0.0280,0.031873,0.900446,0.417474,0.02400,0.03525,0.01125,D3,SECONDARY,DROWSY


In [25]:
train_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/train_dataset_v2_window60.csv",
    index=False
)

test_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/test_dataset_v2_window60.csv",
    index=False
)

print("Window 60 datasets saved successfully!")

Window 60 datasets saved successfully!
